# Lasso-Based Target Selection

Instead of assuming `mental_wellness_index` is the only interesting target, we asked: **which columns in our dataset are most predictable?**

For every continuous column we:
1. Treated it as `y`, all other columns as `X`
2. Fitted `LassoCV` (10-fold) to find the best regularisation
3. Recorded the holdout R², selected features, and regularisation path

## Why Lasso for target screening?

| Property | Benefit |
|---|---|
| L1 penalty zeros weak features | Tells us which features *actually* explain the target |
| Cross-validated alpha | Unbiased estimate of generalisation |
| Sparsity | Few selected features = cleaner, more interpretable model |
| R² score | Direct measure of predictability |

## Results Table

In [ ]:
import pandas as pd
df = pd.read_csv('target_screening_results.csv')
df.sort_values('holdout_R2', ascending=False)

## Figure 1 — Predictability Ranking
`figTS1_target_ranking.png`

- **Green bars** (R² ≥ 0.5) = excellent targets
- **Orange bars** (0.25 ≤ R² < 0.5) = worth studying
- **Red bars** (R² < 0.25) = poor linear targets

The right panel shows how many features Lasso selected — fewer features = cleaner signal.

In [ ]:
from IPython.display import Image
Image('figTS1_target_ranking.png', width=900)

## Figure 2 — Lasso Coefficient Heatmap
`figTS2_lasso_coef_heatmap.png`

Rows = candidate targets (sorted by R²), Columns = features.
- **Red** = positive coefficient (feature increases target)
- **Blue** = negative coefficient
- **White** = coefficient zeroed by Lasso (feature not useful for this target)

This reveals *cross-target patterns*: `age` is selected by almost every target model, making it a universal predictor in this dataset.

In [ ]:
Image('figTS2_lasso_coef_heatmap.png', width=1000)

## Figure 3 — Regularization Paths (Top-4 Targets)
`figTS3_lasso_paths_top4.png`

Each line = one feature. As `alpha` increases (more regularisation), lines collapse to 0.
- Lines that stay non-zero until large alpha = **robust predictors**
- Lines that hit 0 early = **weak or redundant features**

In [ ]:
Image('figTS3_lasso_paths_top4.png', width=950)

## Figure 4 — Target Quality Map
`figTS4_target_quality_map.png`

X-axis = |skewness| of the target; Y-axis = Lasso R²; Colour = # features selected.

**Ideal target** (top-left corner): low skewness + high R² + few features selected.

`mental_wellness_index` and `productivity_score` are the two clear winners, though both are heavily skewed — a **log1p or Box-Cox transform** before modelling is recommended.

In [ ]:
Image('figTS4_target_quality_map.png', width=800)

## Recommendations

| Tier | Targets | Action |
|---|---|---|
| **Excellent** | `mental_wellness_index`, `productivity_score` | Use as primary targets; apply log-transform |
| **Good** | `age`, `mental_health_status`, `online_support_usage`, `support_systems_access`, `stress_level`, `screen_time_hours` | Use as secondary studies |
| **Poor** | `gaming_hours`, `social_media_hours`, `physical_activity_hours`, `social_hours_per_week` | Avoid — not linearly predictable |